In [ ]:
# ============================================================
# SIAMESE CNN FOR GPDS (KERAS 3 SAFE - FINAL)
# ============================================================

import os
import cv2
import re
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.applications import MobileNetV2

# ============================================================
# CONFIG
# ============================================================

GPDS_TRAIN_PATH = r"D:\clg resumes\osv-hybrid (2)\osv-hybrid\training\New folder (10)\train"
IMG_SIZE = 96
EPOCHS = 3
BATCH_SIZE = 32

# ============================================================
# IMAGE LOADER
# ============================================================

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img.reshape(IMG_SIZE, IMG_SIZE, 1)

# ============================================================
# LOAD FILES
# ============================================================

genuine_path = os.path.join(GPDS_TRAIN_PATH, "genuine")
forge_path   = os.path.join(GPDS_TRAIN_PATH, "forge")

genuine_files = os.listdir(genuine_path)
forge_files   = os.listdir(forge_path)

def get_uid(filename):
    m = re.search(r'-(\d{3})-', filename)
    return m.group(1) if m else None

# ============================================================
# BUILD PAIRS
# ============================================================

X1, X2, y = [], [], []

for g in genuine_files:
    uid = get_uid(g)
    if uid is None:
        continue

    pos = [f for f in genuine_files if get_uid(f) == uid and f != g]
    neg = [f for f in forge_files if get_uid(f) == uid]

    if not pos or not neg:
        continue

    g2 = random.choice(pos)
    f2 = random.choice(neg)

    X1.append(load_image(os.path.join(genuine_path, g)))
    X2.append(load_image(os.path.join(genuine_path, g2)))
    y.append(1)

    X1.append(load_image(os.path.join(genuine_path, g)))
    X2.append(load_image(os.path.join(forge_path, f2)))
    y.append(0)

X1 = np.array(X1)
X2 = np.array(X2)
y  = np.array(y).astype("float32")

print("Pairs:", X1.shape, X2.shape)

# ============================================================
# EMBEDDING MODEL (NO CUSTOM LAYERS)
# ============================================================

def embedding_model():
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 1),
        include_top=False,
        weights=None
    )
    for layer in base.layers:
        layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)

    return models.Model(base.input, x)

# ============================================================
# SIAMESE NETWORK (TRAINING ONLY)
# ============================================================

def euclidean_distance(v):
    a, b = v
    return K.sqrt(K.sum(K.square(a - b), axis=1, keepdims=True))

input_a = layers.Input((IMG_SIZE, IMG_SIZE, 1))
input_b = layers.Input((IMG_SIZE, IMG_SIZE, 1))

embed = embedding_model()

ea = embed(input_a)
eb = embed(input_b)

distance = layers.Lambda(euclidean_distance)([ea, eb])

siamese = models.Model([input_a, input_b], distance)

def contrastive_loss(y_true, d):
    margin = 1.0
    return K.mean(
        y_true * K.square(d) +
        (1 - y_true) * K.square(K.maximum(margin - d, 0))
    )

siamese.compile(
    loss=contrastive_loss,
    optimizer=tf.keras.optimizers.Adam(1e-4)
)

# ============================================================
# TRAIN
# ============================================================

siamese.fit(
    [X1, X2],
    y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

print("✅ Siamese trained")

# ============================================================
# SAVE EMBEDDING MODEL (PRODUCTION)
# ============================================================

embedding_net = embedding_model()
embedding_net.set_weights(embed.get_weights())

# 🔥 SAVE AS .keras (NO ERRORS EVER)
embedding_net.save("static_embedding_model.keras")

print("✅ Embedding model saved (static_embedding_model.keras)")


In [ ]:
# ============================================================
# GRAPH 2: DISTANCE DISTRIBUTION
# ============================================================
import matplotlib.pyplot as plt
distances = siamese.predict([X1, X2]).ravel()

genuine_dist = distances[y == 1]
forgery_dist = distances[y == 0]

plt.figure(figsize=(7, 4))
plt.hist(genuine_dist, bins=30, alpha=0.7, label="Genuine", color="green")
plt.hist(forgery_dist, bins=30, alpha=0.7, label="Forgery", color="red")
plt.xlabel("Euclidean Distance")
plt.ylabel("Frequency")
plt.title("Distance Distribution for Genuine and Forged Signatures")
plt.legend()
plt.tight_layout()
plt.savefig("siamese_distance_distribution.png", dpi=300)
plt.show()


In [1]:
import os
import cv2
import re
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.applications import MobileNetV2
from l2_norm_layer import L2Normalization
# ==============================
# CONFIG
# ==============================

GPDS_TRAIN_PATH = r"D:\clg resumes\osv-hybrid (2)\osv-hybrid\training\New folder (10)\train"
IMG_SIZE = 96
EPOCHS = 10
BATCH_SIZE = 32

# ==============================
# LOAD IMAGE (3 CHANNEL NOW)
# ==============================

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # 🔥 Convert grayscale → RGB
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img = img / 255.0
    return img.reshape(IMG_SIZE, IMG_SIZE, 3)


# ==============================
# LOAD FILES
# ==============================

genuine_path = os.path.join(GPDS_TRAIN_PATH, "genuine")
forge_path   = os.path.join(GPDS_TRAIN_PATH, "forge")

genuine_files = os.listdir(genuine_path)
forge_files   = os.listdir(forge_path)

def get_uid(filename):
    m = re.search(r'-(\d{3})-', filename)
    return m.group(1) if m else None

# ==============================
# BUILD PAIRS
# ==============================

X1, X2, y = [], [], []

for g in genuine_files:

    uid = get_uid(g)
    if uid is None:
        continue

    pos = [f for f in genuine_files if get_uid(f) == uid and f != g]
    neg = [f for f in forge_files if get_uid(f) == uid]

    if not pos or not neg:
        continue

    g2 = random.choice(pos)
    f2 = random.choice(neg)

    # positive pair
    X1.append(load_image(os.path.join(genuine_path, g)))
    X2.append(load_image(os.path.join(genuine_path, g2)))
    y.append(1)

    # negative pair
    X1.append(load_image(os.path.join(genuine_path, g)))
    X2.append(load_image(os.path.join(forge_path, f2)))
    y.append(0)

X1 = np.array(X1)
X2 = np.array(X2)
y  = np.array(y).astype("float32")

print("Pairs:", X1.shape)

# ==============================
# EMBEDDING MODEL (FIXED)
# ==============================

def embedding_model():

    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    # freeze first layers
    for layer in base.layers[:-30]:
        layer.trainable = True

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128)(x)
    x = L2Normalization()(x)

    return models.Model(base.input, x)

# ==============================
# SIAMESE NETWORK
# ==============================

def euclidean_distance(v):
    a, b = v
    return K.sqrt(K.sum(K.square(a - b), axis=1, keepdims=True))

input_a = layers.Input((IMG_SIZE, IMG_SIZE, 3))
input_b = layers.Input((IMG_SIZE, IMG_SIZE, 3))

embed = embedding_model()

ea = embed(input_a)
eb = embed(input_b)

distance = layers.Lambda(euclidean_distance)([ea, eb])

siamese = models.Model([input_a, input_b], distance)

def contrastive_loss(y_true, d):
    margin = 1.0
    return K.mean(
        y_true * K.square(d) +
        (1 - y_true) * K.square(K.maximum(margin - d, 0))
    )

siamese.compile(
    loss=contrastive_loss,
    optimizer=tf.keras.optimizers.Adam(1e-4)
)

# ==============================
# TRAIN
# ==============================

siamese.fit(
    [X1, X2],
    y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

print("✅ Siamese trained")

# ==============================
# SAVE ONLY EMBEDDING MODEL
# ==============================

embedding_net = embedding_model()
embedding_net.set_weights(embed.get_weights())

embedding_net.save("static_embedding_model.keras")

print("✅ Embedding model saved")



Pairs: (4800, 96, 96, 3)


Epoch 1/10

150/150 [==============================] - 78s 432ms/step - loss: 0.2581
Epoch 2/10
150/150 [==============================] - 69s 457ms/step - loss: 0.1592
Epoch 3/10
150/150 [==============================] - 68s 451ms/step - loss: 0.1148
Epoch 4/10
150/150 [==============================] - 69s 462ms/step - loss: 0.0790
Epoch 5/10
150/150 [==============================] - 68s 450ms/step - loss: 0.0483
Epoch 6/10
150/150 [==============================] - 71s 477ms/step - loss: 0.0316
Epoch 7/10
150/150 [==============================] - 70s 465ms/step - loss: 0.0227
Epoch 8/10
150/150 [==============================] - 81s 543ms/step - loss: 0.0160
Epoch 9/10
150/150 [==============================] - 71s 470ms/step - loss: 0.0123
Epoch 10/10
150/150 [==============================] - 70s 466ms/step - loss: 0.0106
✅ Siamese trained
✅ Embedding model saved


In [2]:
import os
import cv2
import re
import random
import numpy as np
import tensorflow as tf
from itertools import combinations
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.applications import MobileNetV2

# ==========================================
# CONFIG
# ==========================================

DATA_PATH = r"D:\clg resumes\osv-hybrid (2)\osv-hybrid\training\New folder (10)\train"
IMG_SIZE = 96
BATCH_SIZE = 32
EPOCHS = 5
MARGIN = 2.0
INTER_USER_NEG = 10




from tensorflow.keras.layers import Layer
import tensorflow as tf

class L2Normalize(Layer):
    def call(self, inputs):
        return tf.nn.l2_normalize(inputs, axis=1)

# ==========================================
# IMAGE LOADER
# ==========================================

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img = img.astype("float32") / 255.0
    return img

# ==========================================
# UID EXTRACTOR
# ==========================================

def get_uid(filename):
    name = os.path.splitext(filename)[0]
    m = re.search(r'c-(\d+)-', name.lower())
    if m:
        return m.group(1).zfill(3)
    return None

# ==========================================
# LOAD FILES
# ==========================================

genuine_path = os.path.join(DATA_PATH, "genuine")
forge_path   = os.path.join(DATA_PATH, "forge")

genuine_files = os.listdir(genuine_path)
forge_files   = os.listdir(forge_path)

# ==========================================
# GROUP BY USER
# ==========================================

user_genuine = {}
user_forge = {}

for f in genuine_files:
    uid = get_uid(f)
    if uid:
        user_genuine.setdefault(uid, []).append(f)

for f in forge_files:
    uid = get_uid(f)
    if uid:
        user_forge.setdefault(uid, []).append(f)

print("Total Users:", len(user_genuine))

# ==========================================
# BUILD PAIRS (ONLY PATHS, NOT IMAGES)
# ==========================================

pairs = []
users = list(user_genuine.keys())

for uid in users:

    g_list = user_genuine[uid]
    f_list = user_forge.get(uid, [])

    # Genuine-Genuine (positive)
    for g1, g2 in combinations(g_list, 2):
        pairs.append((
            os.path.join(genuine_path, g1),
            os.path.join(genuine_path, g2),
            1
        ))

    # Genuine-Forgery (negative)
    for g in g_list:
        for f in f_list:
            pairs.append((
                os.path.join(genuine_path, g),
                os.path.join(forge_path, f),
                0
            ))

# Inter-user negatives (limited)
for _ in range(len(users) * INTER_USER_NEG):
    u1, u2 = random.sample(users, 2)
    g1 = random.choice(user_genuine[u1])
    g2 = random.choice(user_genuine[u2])

    pairs.append((
        os.path.join(genuine_path, g1),
        os.path.join(genuine_path, g2),
        0
    ))

print("Total Pairs:", len(pairs))

# ==========================================
# SPLIT
# ==========================================

train_pairs, val_pairs = train_test_split(
    pairs, test_size=0.2, random_state=42
)

# ==========================================
# DATA GENERATOR
# ==========================================

class SiameseGenerator(tf.keras.utils.Sequence):

    def __init__(self, pairs, batch_size=32):
        self.pairs = pairs
        self.batch_size = batch_size

    def __len__(self):
        return len(self.pairs) // self.batch_size

    def __getitem__(self, idx):

        batch = self.pairs[idx * self.batch_size:(idx + 1) * self.batch_size]

        X1, X2, y = [], [], []

        for p1, p2, label in batch:
            img1 = load_image(p1)
            img2 = load_image(p2)

            X1.append(img1)
            X2.append(img2)
            y.append(label)

        return (
            [np.array(X1, dtype=np.float32),
             np.array(X2, dtype=np.float32)],
            np.array(y, dtype=np.float32)
        )

train_gen = SiameseGenerator(train_pairs, BATCH_SIZE)
val_gen   = SiameseGenerator(val_pairs, BATCH_SIZE)

# ==========================================
# EMBEDDING MODEL
# ==========================================

def build_embedding():

    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    for layer in base.layers:
        layer.trainable = False

    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128)(x)

    # 🔥 SAFE NORMALIZATION
    x = L2Normalize()(x)

    return tf.keras.Model(base.input, x)

# ==========================================
# SIAMESE NETWORK
# ==========================================

def euclidean_distance(v):
    a, b = v
    return K.sqrt(K.sum(K.square(a - b), axis=1, keepdims=True))

input_a = layers.Input((IMG_SIZE, IMG_SIZE, 3))
input_b = layers.Input((IMG_SIZE, IMG_SIZE, 3))

embedding_net = build_embedding()

ea = embedding_net(input_a)
eb = embedding_net(input_b)

distance = layers.Lambda(euclidean_distance)([ea, eb])

siamese = models.Model([input_a, input_b], distance)

# ==========================================
# LOSS
# ==========================================

def contrastive_loss(y_true, d):
    return K.mean(
        y_true * K.square(d) +
        (1 - y_true) * K.square(K.maximum(MARGIN - d, 0))
    )

siamese.compile(
    loss=contrastive_loss,
    optimizer=tf.keras.optimizers.Adam(1e-4)
)

# ==========================================
# TRAIN
# ==========================================

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

siamese.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

print("✅ Training Completed")

# ==========================================
# SAVE EMBEDDING MODEL
# ==========================================

embedding_net.save("final_signature_embedding_model.keras")
print("✅ Embedding Model Saved")

Total Users: 150
Total Pairs: 19500
Epoch 1/5
487/487 [==============================] - 100s 199ms/step - loss: 0.3691 - val_loss: 0.2738
Epoch 2/5
487/487 [==============================] - 100s 205ms/step - loss: 0.2971 - val_loss: 0.2770
Epoch 3/5
487/487 [==============================] - 101s 207ms/step - loss: 0.2906 - val_loss: 0.2771
Epoch 4/5
487/487 [==============================] - 138s 284ms/step - loss: 0.2877 - val_loss: 0.2736
Epoch 5/5
487/487 [==============================] - 106s 218ms/step - loss: 0.2841 - val_loss: 0.2588
✅ Training Completed
✅ Embedding Model Saved


In [1]:
import os
import cv2
import re
import random
import numpy as np
import tensorflow as tf
from itertools import combinations
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.applications import MobileNetV2

# ==========================================
# CONFIG
# ==========================================

DATA_PATH = r"C:\Users\Nandini\OneDrive\Desktop\osv-hybrid (1)\osv-hybrid\training\New folder (10)\train"
IMG_SIZE = 96
BATCH_SIZE = 32
EPOCHS =30         # 🔥 increase
MARGIN = 1.0         # 🔥 better margin
INTER_USER_NEG = 20  # 🔥 more inter-user negatives

# ==========================================
# SAFE L2 NORMALIZATION
# ==========================================

class L2Normalize(layers.Layer):
    def call(self, inputs):
        return tf.nn.l2_normalize(inputs, axis=1)

# ==========================================
# IMAGE LOADER
# ==========================================

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img = img.astype("float32") / 255.0
    return img

# ==========================================
# UID EXTRACTOR
# ==========================================

def get_uid(filename):
    name = os.path.splitext(filename)[0]
    m = re.search(r'c-(\d+)-', name.lower())
    if m:
        return m.group(1).zfill(3)
    return None

# ==========================================
# LOAD FILES
# ==========================================

genuine_path = os.path.join(DATA_PATH, "genuine")
forge_path   = os.path.join(DATA_PATH, "forge")

genuine_files = os.listdir(genuine_path)
forge_files   = os.listdir(forge_path)

# ==========================================
# GROUP BY USER
# ==========================================

user_genuine = {}
user_forge = {}

for f in genuine_files:
    uid = get_uid(f)
    if uid:
        user_genuine.setdefault(uid, []).append(f)

for f in forge_files:
    uid = get_uid(f)
    if uid:
        user_forge.setdefault(uid, []).append(f)

print("Total Users:", len(user_genuine))

# ==========================================
# BUILD PAIRS
# ==========================================

pairs = []
users = list(user_genuine.keys())

for uid in users:

    g_list = user_genuine[uid]
    f_list = user_forge.get(uid, [])

    # Genuine-Genuine (positive)
    for g1, g2 in combinations(g_list, 2):
        pairs.append((
            os.path.join(genuine_path, g1),
            os.path.join(genuine_path, g2),
            1
        ))

    # Genuine-Forgery (negative)
    for g in g_list:
        for f in f_list:
            pairs.append((
                os.path.join(genuine_path, g),
                os.path.join(forge_path, f),
                0
            ))

# 🔥 Stronger inter-user negatives
for _ in range(len(users) * INTER_USER_NEG):
    u1, u2 = random.sample(users, 2)
    g1 = random.choice(user_genuine[u1])
    g2 = random.choice(user_genuine[u2])

    pairs.append((
        os.path.join(genuine_path, g1),
        os.path.join(genuine_path, g2),
        0
    ))

print("Total Pairs:", len(pairs))

# ==========================================
# SPLIT
# ==========================================

train_pairs, val_pairs = train_test_split(
    pairs, test_size=0.2, random_state=42
)

# ==========================================
# GENERATOR
# ==========================================

class SiameseGenerator(tf.keras.utils.Sequence):

    def __init__(self, pairs, batch_size=32):
        self.pairs = pairs
        self.batch_size = batch_size

    def __len__(self):
        return len(self.pairs) // self.batch_size

    def on_epoch_end(self):
        random.shuffle(self.pairs)

    def __getitem__(self, idx):

        batch = self.pairs[idx * self.batch_size:(idx + 1) * self.batch_size]

        X1, X2, y = [], [], []

        for p1, p2, label in batch:
            X1.append(load_image(p1))
            X2.append(load_image(p2))
            y.append(label)

        return (
    (np.array(X1, dtype=np.float32),
     np.array(X2, dtype=np.float32)),
    np.array(y, dtype=np.float32)
)

train_gen = SiameseGenerator(train_pairs, BATCH_SIZE)
val_gen   = SiameseGenerator(val_pairs, BATCH_SIZE)

# ==========================================
# EMBEDDING MODEL
# ==========================================

def build_embedding():

    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    # 🔥 fine tune last layers only
    for layer in base.layers[:-30]:
        layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128)(x)
    x = L2Normalize()(x)

    return models.Model(base.input, x)

# ==========================================
# SIAMESE NETWORK
# ==========================================

def euclidean_distance(v):
    a, b = v
    return K.sqrt(K.sum(K.square(a - b), axis=1, keepdims=True))

input_a = layers.Input((IMG_SIZE, IMG_SIZE, 3))
input_b = layers.Input((IMG_SIZE, IMG_SIZE, 3))

embedding_net = build_embedding()

ea = embedding_net(input_a)
eb = embedding_net(input_b)

distance = layers.Lambda(euclidean_distance)([ea, eb])

siamese = models.Model([input_a, input_b], distance)

# ==========================================
# LOSS
# ==========================================

def contrastive_loss(y_true, d):
    return K.mean(
        y_true * K.square(d) +
        (1 - y_true) * K.square(K.maximum(MARGIN - d, 0))
    )

siamese.compile(
    loss=contrastive_loss,
    optimizer=tf.keras.optimizers.Adam(1e-4)
)

# ==========================================
# TRAIN
# ==========================================

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

siamese.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

print("✅ Training Completed")

# ==========================================
# SAVE EMBEDDING MODEL
# ==========================================

embedding_net.save("final_signature_embedding_model.keras")
print("✅ Embedding Model Saved")

Total Users: 150
Total Pairs: 21000



C:\Users\Nandini\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 170s 303ms/step - loss: 0.1941 - val_loss: 0.1182
Epoch 2/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 438s 834ms/step - loss: 0.1233 - val_loss: 0.0875
Epoch 3/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 415s 790ms/step - loss: 0.0952 - val_loss: 0.0634
Epoch 4/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 402s 765ms/step - loss: 0.0750 - val_loss: 0.0531
Epoch 5/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 147s 280ms/step - loss: 0.0621 - val_loss: 0.0481
Epoch 6/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 151s 287ms/step - loss: 0.0535 - val_loss: 0.0430
Epoch 7/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 169s 322ms/step - loss: 0.0470 - val_loss: 0.0412
Epoch 8/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 636s 1s/step - loss: 0.0434 - val_loss: 0.0406
Epoch 9/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 148s 281ms/step - loss: 0.0393 - val_loss: 0.0360
Epoch 10/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 152s 290ms/step - loss: 0.0357 - val_loss: 0.0328
Epoch 11/30
525/525 ━━━━━━━━━━━━━━━━━━━━ 155s 296ms/step - loss: 0.0361 - val_loss: 0.0332
Epoch 12/30

In [3]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import numpy as np

# Collect validation predictions
y_true = []
y_pred = []
distances = []

for (X1, X2), labels in val_gen:
    d = siamese.predict([X1, X2])
    distances.extend(d.flatten())
    y_true.extend(labels)

y_true = np.array(y_true)
distances = np.array(distances)

# 🔥 Find best threshold using ROC logic
best_acc = 0
best_thresh = 0

for t in np.linspace(0, 2, 200):
    preds = (distances < t).astype(int)
    acc = accuracy_score(y_true, preds)
    if acc > best_acc:
        best_acc = acc
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best Accuracy:", best_acc)

# Final predictions
final_preds = (distances < best_thresh).astype(int)

print(confusion_matrix(y_true, final_preds))
print(classification_report(y_true, final_preds))
print("AUC:", roc_auc_score(y_true, -distances))

NameError: name 'val_gen' is not defined